# Laboratorio # 5

Vamos a leer el dataset

In [6]:
import pandas as pd


train = pd.read_csv("test.csv")
test = pd.read_csv("train.csv")


In [7]:
train.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [8]:
test.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Limpieza

### Quitar los signos de puntuacion

In [10]:
import re

train["text"] = train["text"].str.replace(r'[^\w\s]', '', regex=True)

test["text"] = test["text"].str.replace(r'[^\w\s]', '', regex=True)


In [11]:
test.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this earthquake Ma...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask Canada,1
2,5,NaN,NaN,All residents asked to shelter in place are be...,1
3,6,NaN,NaN,13000 people receive wildfires evacuation orde...,1
4,7,NaN,NaN,Just got sent this photo from Ruby Alaska as s...,1


### Quitar los artículos, preposiciones y conjunciones



In [12]:
import pandas as pd
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    tokens = text.split()  # separar en palabras
    tokens = [word for word in tokens if word.lower() not in stop_words]
    return " ".join(tokens)

# Aplicar a las columnas
train["text"] = train["text"].apply(remove_stopwords)
test["text"] = test["text"].apply(remove_stopwords)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mathew\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [13]:
test.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Deeds Reason earthquake May ALLAH Forgive us,1
1,4,NaN,NaN,Forest fire near La Ronge Sask Canada,1
2,5,NaN,NaN,residents asked shelter place notified officer...,1
3,6,NaN,NaN,13000 people receive wildfires evacuation orde...,1
4,7,NaN,NaN,got sent photo Ruby Alaska smoke wildfires pou...,1


### Quitar numeros

Primero vamos a ver cuales eliminar, listando por los que mas aparecen

In [16]:
import re
import pandas as pd

def extract_numbers(text):
    return re.findall(r'\d+', str(text))
train_nums = train.copy()
train_nums["numbers"] = train_nums["text"].apply(extract_numbers)

test_nums = test.copy()
test_nums["numbers"] = test_nums["text"].apply(extract_numbers)

all_nums = pd.concat([train_nums[["numbers"]], test_nums[["numbers"]]], ignore_index=True)

all_nums = all_nums[all_nums["numbers"].map(len) > 0]

import numpy as np
all_numbers_flat = np.concatenate(all_nums["numbers"].values)

summary = pd.Series(all_numbers_flat).value_counts().reset_index()
summary.columns = ["number", "count"]


In [17]:
summary

,number,count
0,2,1189
1,3,1090
2,1,1061
3,5,1028
4,4,1017
...,...,...
930,080215,1
931,29916207,1
932,463,1
933,1942,1


Sabiendo esto vamos a conservar los numeros 1945, 911, 2008, 2014, 1980, 2013, 2016, 2011  ya que son fechas importantes, ademas de que son numeros de telefono que pueden tener relacion con el analisis de sentimientos

In [18]:
import re

allowed_numbers = {"1945", "911", "2008", "2014", "1980", "2013", "2016", "2011"}

def remove_unwanted_numbers(text):
    return re.sub(r'\b(?!' + '|'.join(allowed_numbers) + r')\d+\b', '', str(text))

train["text"] = train["text"].apply(remove_unwanted_numbers)
test["text"] = test["text"].apply(remove_unwanted_numbers)


In [19]:
test.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Deeds Reason earthquake May ALLAH Forgive us,1
1,4,NaN,NaN,Forest fire near La Ronge Sask Canada,1
2,5,NaN,NaN,residents asked shelter place notified officer...,1
3,6,NaN,NaN,people receive wildfires evacuation orders Ca...,1
4,7,NaN,NaN,got sent photo Ruby Alaska smoke wildfires pou...,1
